thank you to chatgpt

In [2]:
import numpy as np
import os
import matplotlib.pyplot as plt

In [1]:
def b92_bit(ch1, ch2):
    """
    Map photon detector counts to Bob's bit according to B92 protocol.
    ch1: counts for |90°> detector (testing for 0)
    ch2: counts for |-45°> detector (testing for 1)
    Returns 0, 1, or None (inconclusive)
    """
    if ch1 > 0 and ch2 == 0:
        return 0
    elif ch2 > 0 and ch1 == 0:
        return 1
    else:
        return -1 # inconclusive, discard

In [3]:
def create_sifted_index_arrays(repeat_array, N_duplicate):
    """
    Creates an array of indices, where each value corresponds to the original index of a duplicate bit.
    E.g.
        Original string: 1 0 1 (indices 0,1,2)
        Repeat string (N=3): 111 000 111
        Received string: 1XX 0X1 XX0

        index array: [0, 1,1, 2] (corresponds to original bit)
    
    Inputs:
        repeat_array: received bit string
        N_duplicate: how many times each bit duplicated
    
    Outputs:
        sifted_array: array of information received
        index_array: corresponding indices
    """

    sifted_array = []
    index_array = []
    for i in range(len(repeat_array)):
        if repeat_array[i]!= -1:
            sifted_array.append(repeat_array[i])
            index_array.append(i//N_duplicate)
    
    return np.array(sifted_array), np.array(index_array)

def calculate_parity(sifted_array):
    """
    Calculates parity of a subarray.
    parity = sum(array) mod2

    Input: 
        - sifted_array : subarray to calculate parity of
    Output: 
        - p : parity
    """
    return np.sum(sifted_array)%2

def locate_error(A, B, offset=0):
    """ 
    Locates an error between Alice & Bob using a recursive binary search.
    Assumes parity(A) != parity(B), i.e., there is an error somewhere.

    Inputs:
        - A: Alice's sifted subarray
        - B: Bob's corresponding subarray
        - offset: starting index of this subarray in the original array

    Output:
        error_index: index in the original subarray where Bob has an error
    """
    if len(A) == 1:
        if A[0] != B[0]:
            return offset
        else:
            return None  # No error here (shouldn't happen if called correctly)
    
    mid = len(A) // 2
    # Check parity of the first half
    if calculate_parity(A[:mid]) != calculate_parity(B[:mid]):
        return locate_error(A[:mid], B[:mid], offset=offset)
    else:
        return locate_error(A[mid:], B[mid:], offset=offset + mid)




In [9]:
locate_error([1,0,1,1],[1,0,0,1])

2

In [17]:
import numpy as np

def cascade_correct(A, B, n_passes=4, block_size_init=8):
    """
    Basic (non-optimized) Cascade protocol.
    
    Inputs:
        A: Alice's bit array
        B: Bob's bit array (will be corrected)
        n_passes: number of Cascade passes
        block_size_init: initial block size
    
    Output:
        B_corrected
    """
    B = B.copy()
    N = len(A)

    def parity(arr):
        return np.sum(arr) % 2

    def binary_search_error(A_sub, B_sub, indices):
        """Locate a single error using recursive parity checks."""
        if len(indices) == 1:
            return indices[0]

        mid = len(indices) // 2
        left_idx = indices[:mid]
        right_idx = indices[mid:]

        if parity(A[left_idx]) != parity(B[left_idx]):
            return binary_search_error(A, B, left_idx)
        else:
            return binary_search_error(A, B, right_idx)

    for p in range(n_passes):
        # Increase block size each pass
        block_size = block_size_init * (2 ** p)

        # Shuffle indices (key part of Cascade)
        perm = np.random.permutation(N)
        
        # Process blocks
        for i in range(0, N, block_size):
            block = perm[i:i+block_size]

            if len(block) == 0:
                continue

            if parity(A[block]) != parity(B[block]):
                # Locate and fix error
                err_idx = binary_search_error(A, B, block)
                B[err_idx] = (B[err_idx]+1)%2  # flip bit

    return B

In [50]:
A = np.zeros(2048)
B = np.zeros(2048)

for i in range(len(B)):
    A[i] = np.random.randint(0,2)
    B[i] = A[i] + np.random.randint(0,51)//50

print(B[0:100])

[0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1.
 0. 1. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 0. 0. 1. 1. 1.
 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 2. 1. 1. 1. 0. 0. 0. 1. 1. 0.
 0. 0. 0. 0.]


In [52]:
def pick_npass_k1(length, error_rate):
    k1 = int(1 / error_rate)
    k1 = max(4, min(k1, length))
    return k1

In [56]:
k = pick_npass_k1(2048, .02)
B_corr = cascade_correct(A,B,n_passes=3, block_size_init=k)
print(A[0:100] - B_corr[0:100])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]
